# Running an unlearning evaluation with SUPREME

SUPREME automates the train -> unlearn -> evaluate pipeline used to compare machine-unlearning methods. The paper demonstrates the framework by evaluating several unlearning methods on the Pins Face Recognition dataset with ResNet18 and ViT, under full-class and random-sample unlearning, across ten training seeds.

This notebook is a tutorial for running that kind of evaluation. It covers installation, credential configuration, a single-cell check of the pipeline, launching the evaluation grid, and rendering the result tables. The demonstration from the paper is used as the worked example; the same commands apply to any combination of the framework's datasets, models, methods, scenarios, and seeds. The companion reference is [`docs/reproducing_the_paper.md`](../docs/reproducing_the_paper.md).

> **Hardware.** The pipeline needs a GPU (NVIDIA CUDA or Apple Silicon MPS) for realistic runtimes. The paper's demonstration ran on a **single NVIDIA L40S (48 GB)**. Multi-GPU runs are not bit-equivalent to single-GPU runs because of distributed gradient averaging, so a single-GPU run is the closest match to the reported numbers. The shell cells below assume the notebook runs from a clone of the repository with the environment activated.

> **Where results appear.** The run cells in sections 4 and 5 launch the pipeline as a detached background job. Their detailed progress is streamed to a log file under `logs/output_log_files/` (git-ignored, so it is not committed with this notebook), and the metrics are recorded to Weights & Biases. Those cells therefore show only a launch confirmation, not inline accuracy or loss numbers. The reproduced results are the LaTeX tables rendered in section 6 from the W&B export.

## 1. Install

Clone the repository and install the framework. The tagged release [`v0.1.0-paper`](https://github.com/pedroandreou/supreme-unlearning/releases/tag/v0.1.0-paper) is the reference point for the numbers reported in the paper; later commits keep the behaviour, defaults, and seeds identical.

In [ ]:
# One-time setup - run this first. The Makefile is the single entry point for all
# environment setup in this repo (venv + pinned deps + editable install + git
# hook), so it is invoked here rather than calling pip directly.
#
# In a shell, clone first (and optionally pin the paper tag):
#   git clone https://github.com/pedroandreou/supreme-unlearning.git
#   cd supreme-unlearning && git checkout v0.1.0-paper   # optional
import subprocess, platform, pathlib

# Find the repo root (holds the Makefile/pyproject.toml) from the kernel's cwd.
root = pathlib.Path.cwd()
while root != root.parent and not (root / "Makefile").exists():
    root = root.parent
assert (root / "Makefile").exists(), f"repo root not found from {pathlib.Path.cwd()}"

# Linux + NVIDIA -> `make cuda`; Apple Silicon / CPU / other -> `make mps`.
target = "cuda" if platform.system() == "Linux" else "mps"
print("repo root:", root, "| running: make", target)

# ON_EXISTING=reuse keeps it non-interactive (reinstall into the existing venv).
subprocess.run(["make", target, "ON_EXISTING=reuse"], cwd=root, check=True)
print("done. The Makefile installs into the venv (default: unlearning). The")
print("notebook kernel must be that venv (restart and pick it from the selector).")

## 2. Configure credentials

Evaluation results are logged to Weights & Biases, and ViT weights are pulled from the Hugging Face Hub, so both tokens are required. Copy the template and fill in the keys (see [`docs/environment_setup.md`](../docs/environment_setup.md)).

In [1]:
# Load credentials from the repo-root .env. First time: copy and fill it in:
#   cp .env.example .env   # then set WANDB_KEY, WANDB_USERNAME, HUGGING_FACE_HUB_TOKEN
from dotenv import load_dotenv
import os, pathlib

# Resolve the repo root so this works regardless of the kernel's cwd.
_root = globals().get("root")
if _root is None:
    _root = pathlib.Path.cwd()
    while _root != _root.parent and not (_root / "pyproject.toml").exists():
        _root = _root.parent
load_dotenv(_root / ".env", override=True)

# Check the names the framework + .env.example actually use (NOT WANDB_API_KEY / HF_TOKEN):
#   W&B key  -> WANDB_KEY  (the framework also accepts WANDB_API_KEY as a fallback)
#   HF token -> HUGGING_FACE_HUB_TOKEN  (auto-detected by huggingface_hub)
wandb_key = os.getenv("WANDB_KEY") or os.getenv("WANDB_API_KEY")
print("W&B key set      :", bool(wandb_key))
print("W&B username set :", bool(os.getenv("WANDB_USERNAME")))
print("HF token set     :", bool(os.getenv("HUGGING_FACE_HUB_TOKEN")))

W&B key set      : True
W&B username set : True
HF token set     : True


## 3. Verify the install

`import supreme` is lightweight (it does not import the PyTorch/Lightning stack),
so it is a quick check that the package resolves and exposes its public API.

In [2]:
import supreme

print("SUPREME version:", supreme.__version__)
print("Public API:", [n for n in supreme.__all__ if n.startswith(("register", "run"))])

# Built-in components available out of the box:
print("Models   :", supreme.project_config.model_names)
print("Methods  :", supreme.project_config.all_methods)
print("Datasets :", supreme.project_config.dataset_names)
print("Metrics  :", supreme.project_config.evaluation_metrics)

SUPREME version: 0.1.2.dev0
Public API: ['register_model', 'register_baseline', 'register_unlearning_method', 'register_metric', 'register_dataset', 'run_training', 'run_unlearning']
Models   : ['ResNet18', 'ViT']
Methods  : ['original', 'retrain', 'finetune', 'bad_teacher', 'random_labeling', 'unsir', 'ssd', 'lfssd', 'assd', 'scrub', 'jit']
Datasets : ['Cifar10', 'Cifar20', 'Cifar100', 'PinsFaceRecognition', 'Caltech101']
Metrics  : ['accuracy', 'activation_distance', 'completeness', 'jsdiv', 'layerwise_distance', 'membership_inference_attack', 'resource_consumption', 'time', 'zrf']


## 4. Single-cell check

Before the full grid, the end-to-end **train -> unlearn -> evaluate** pipeline can be confirmed on one small cell. Re-running is safe: training checkpoints, unlearning checkpoints, and already-logged W&B results are detected and skipped. There are two equivalent ways to launch it.

In [3]:
# (a) Single-cell check via the experiment-grid driver. Two Jupyter constraints
# apply here: large cell outputs get truncated, and the `!cmd &` shell magic
# raises "Background processes not supported". So the command is launched with
# subprocess in the background, with full output redirected to a .log under
# logs/output_log_files/ (open it in an editor to follow progress). start_new_session
# detaches the process so it survives a kernel restart. `root` comes from section 1.
#
# Runs on any OS (auto-selects MPS/CPU); the paper's numbers need Linux + NVIDIA GPU
# + CUDA. The pipeline's file locks are handled portably, so this also runs on macOS.
import subprocess

logdir = root / "logs" / "output_log_files"
logdir.mkdir(parents=True, exist_ok=True)
smoke_log = logdir / "smoke_test.log"

cmd = [
    "bash",
    str(root / "supreme" / "run_local.sh"),
    "--gpu",
    "0",
    "--models",
    "ViT",
    "--training-seeds",
    "260",
    "--methods",
    "retrain,finetune,ssd",
    "--strategies",
    "random_",
    "--datasets",
    "Cifar10",
    "--forget-percs",
    "0.01",
]
with open(smoke_log, "w") as f:
    proc = subprocess.Popen(
        cmd, cwd=str(root), stdout=f, stderr=subprocess.STDOUT, start_new_session=True
    )
print(f"Single-cell check launched in the background (PID {proc.pid}). Full log:")
print("  ", smoke_log)
print(f"Check progress:  !tail -n 80 {smoke_log}")

Single-cell check launched in the background (PID 27860). Full log:
   ~/supreme-unlearning/logs/output_log_files/smoke_test.log
Check progress:  !tail -n 80 ~/supreme-unlearning/logs/output_log_files/smoke_test.log


In [ ]:
# (b) The Python API for a SINGLE unlearning stage - same arguments as the
#     `supreme-unlearn` console script.
#
#     -weight_path is normally not set by hand. The driver in (a) discovers the
#     trained checkpoint automatically: src/supreme/MAIN.sh -> find_checkpoint() calls
#     src/supreme/utils/unlearning/update_checkpoint_paths.py, which finds the latest
#     *-best.pth (gated on a TRAINING_DONE marker) and passes it as -weight_path.
#     This direct call is only for targeting one specific checkpoint.
# import supreme
# supreme.run_unlearning([
#     '-method', 'ssd', '-net', 'ViT', '-dataset', 'Cifar10',
#     '-type_of_unlearning_strategy', 'random_', '-seed', '260',
#     '-precision', '32-true', '-weight_path', '<path/to/trained.ckpt>',
# ])

## 5. Run the evaluation grid

The command below configures the evaluation used in the paper's demonstration: six unlearning methods plus the retrain baseline, both scenarios (`fullclass`, `random_`), both architectures, ten training seeds (260-269), and a 0.1% forget set for the random-sample scenario. Changing these flags runs a different evaluation.

**Matched-seed protocol (`J = K = 1`, the default).** For each training seed, the unlearning and evaluation seeds are tied to it: `s_t = s_u = s_e`. This is selected by `--unlearning-seeds "0" --evaluation-seeds "0"` (passed explicitly below, though they are the defaults). The seed-protocol table is in [`src/supreme/README.md`](../src/supreme/README.md) and [`docs/notation.md`](../docs/notation.md); the decoupled protocols (`J > 1` and/or `K > 1`) are for variance studies.

This is a long-running job. It runs on the GPU machine and populates W&B. Interruptions are safe to resume, since finished cells are skipped. Because it can run for hours, the cell below launches it in the background and redirects output to `logs/output_log_files/repro_grid.log`, so the cell returns immediately and the job survives a kernel restart. The next cell tails that log and can be re-run at any time to check progress.

Pins Face Recognition requires a manual Kaggle download (see [`docs/adding_pinsfacerecognition.md`](../docs/adding_pinsfacerecognition.md)).

In [4]:
# Run the full grid in the background via subprocess (Jupyter's `!cmd &` magic
# raises "Background processes not supported"), with full output redirected to a
# .log under logs/output_log_files/. start_new_session detaches the process so it
# survives a kernel restart. `root` comes from section 1.
#
# Matched-seed protocol (J=K=1): 10 training seeds (260-269), and for each one
# s_t = s_u = s_e. The --unlearning-seeds/--evaluation-seeds "0" flags set this
# explicitly (they are also the defaults).
#
# Runs on any OS, but the paper's numbers require Linux + NVIDIA GPU + CUDA. On a
# macOS/CPU host this trains on MPS/CPU and is too slow for the full grid.
# PinsFaceRecognition also needs a manual Kaggle download first.
import subprocess

logdir = root / "logs" / "output_log_files"
logdir.mkdir(parents=True, exist_ok=True)
logfile = logdir / "repro_grid.log"

cmd = [
    "bash",
    str(root / "supreme" / "run_local.sh"),
    "--gpu",
    "0",
    "--datasets",
    "PinsFaceRecognition",
    "--models",
    "ResNet18,ViT",
    "--strategies",
    "fullclass,random_",
    "--methods",
    "retrain,finetune,bad_teacher,random_labeling,unsir,ssd,lfssd",
    "--forget-percs",
    "0.001",
    "--training-seeds",
    "260,261,262,263,264,265,266,267,268,269",
    "--unlearning-seeds",
    "0",
    "--evaluation-seeds",
    "0",
]
with open(logfile, "w") as f:
    proc = subprocess.Popen(
        cmd, cwd=str(root), stdout=f, stderr=subprocess.STDOUT, start_new_session=True
    )
print(f"Launched in the background (PID {proc.pid}). Log:", logfile)
print("Monitor it with the next cell (re-run anytime).")

Launched in the background (PID 27861). Log: ~/supreme-unlearning/logs/output_log_files/repro_grid.log
Monitor it with the next cell (re-run anytime).


In [ ]:
# Monitor the background grid run (re-run anytime; the job keeps going).
!tail -n 50 {logfile}

# Is it still running?   ->  !pgrep -fl run_local.sh
# Stop it early?         ->  !pkill -f run_local.sh

## 6. Render the result tables

Once the grid has populated W&B, the metrics are exported and the three LaTeX tables from the paper are rendered with [`src/supreme/utils/wandb_utils/results_analysis/pins_paper_tables.ipynb`](../src/supreme/utils/wandb_utils/results_analysis/pins_paper_tables.ipynb). The export-to-LaTeX workflow and troubleshooting notes are in [`docs/reproducing_the_paper.md`](../docs/reproducing_the_paper.md) and [`docs/tooling.md`](../docs/tooling.md).